In [3]:
from dataclasses import dataclass, field
from enum import Enum
import re
import time
from typing import List, Optional
import spacy


class ReasoningEffort(Enum):
    NONE = "none"
    LOW = "low"
    HIGH = "high"


@dataclass
class StageLatency:
    spacy_doc_parse_ms: float = 0.0
    direct_intent_eval_ms: float = 0.0
    syntactic_complexity_ms: float = 0.0
    domain_entity_eval_ms: float = 0.0
    total_ms: float = 0.0


@dataclass
class RoutingDecision:
    query: str
    requires_thinking: bool
    effort_level: ReasoningEffort
    matched_reasons: List[str]
    confidence: float
    latencies: StageLatency = field(default_factory=StageLatency)


class SpacyComplexityRouter:

    def __init__(self, spacy_model: str = "en_core_web_sm"):
        # Load spaCy pipeline; disable unneeded components for speed
        self.nlp = spacy.load(
            spacy_model,
            exclude=["ner"]  # Re-enable if entity classification is needed
        )

        # Mathematical and algorithmic vocabulary lemmas
        self.reasoning_lemmas = {
            "prove", "derive", "calculate", "compute", "optimize", "analyze",
            "debug", "solve", "evaluate", "compare", "contrast", "deduce",
            "simulate", "balance", "refactor"
        }

        # Pure generation/bypass verbs
        self.bypass_verbs = {"translate", "reformat", "summarize", "paraphrase", "greet"}
        self.nlp("warmup query")

    def analyze(self, query: str) -> RoutingDecision:
        total_start = time.perf_counter()
        latencies = StageLatency()
        matched_reasons = []

        # -------------------------------------------------------------
        # STAGE 1: spaCy Tokenization & Linguistic Parsing
        # -------------------------------------------------------------
        parse_start = time.perf_counter()
        doc = self.nlp(query)
        parse_end = time.perf_counter()
        latencies.spacy_doc_parse_ms = (parse_end - parse_start) * 1000

        # Fast path: Empty or single greeting token
        if len(doc) <= 2 and any(token.lower_ in {"hi", "hello", "hey", "thanks", "bye"} for token in doc):
            latencies.total_ms = (time.perf_counter() - total_start) * 1000
            return RoutingDecision(
                query=query,
                requires_thinking=False,
                effort_level=ReasoningEffort.NONE,
                matched_reasons=["salutation_token"],
                confidence=0.99,
                latencies=latencies,
            )

        # -------------------------------------------------------------
        # STAGE 2: Verb & Intent Extraction (Root Verb Analysis)
        # -------------------------------------------------------------
        intent_start = time.perf_counter()
        root_token = [token for token in doc if token.head == token]
        root_verb = root_token[0].lemma_.lower() if root_token else ""

        # Check for reasoning-triggering lemmas across the entire token set
        lemma_matches = [
            token.lemma_.lower()
            for token in doc
            if token.lemma_.lower() in self.reasoning_lemmas
        ]
        if lemma_matches:
            matched_reasons.append(f"reasoning_lemmas({','.join(lemma_matches)})")

        # Check for direct bypass verbs at the sentence root
        if root_verb in self.bypass_verbs and not lemma_matches:
            matched_reasons.append(f"direct_root_intent({root_verb})")
            intent_end = time.perf_counter()
            latencies.direct_intent_eval_ms = (intent_end - intent_start) * 1000
            latencies.total_ms = (intent_end - total_start) * 1000
            return RoutingDecision(
                query=query,
                requires_thinking=False,
                effort_level=ReasoningEffort.NONE,
                matched_reasons=matched_reasons,
                confidence=0.92,
                latencies=latencies,
            )

        intent_end = time.perf_counter()
        latencies.direct_intent_eval_ms = (intent_end - intent_start) * 1000

        # -------------------------------------------------------------
        # STAGE 3: Syntactic Complexity & Dependency Depth
        # -------------------------------------------------------------
        syntax_start = time.perf_counter()

        # Depth heuristic: measure dependency tree height
        def get_depth(token):
            return 1 + max((get_depth(child) for child in token.children), default=0)

        max_tree_depth = max((get_depth(token) for token in doc if token.head == token), default=0)

        # Subordinate / conditional clauses (e.g., "if", "unless", "assuming that", "whereas")
        sub_clauses = [token for token in doc if token.dep_ in {"advcl", "ccomp", "xcomp"}]
        conditionals = [token for token in doc if token.lower_ in {"if", "assuming", "suppose", "given"}]

        if max_tree_depth >= 5:
            matched_reasons.append(f"deep_syntactic_tree(depth={max_tree_depth})")

        if conditionals:
            matched_reasons.append("conditional_clause_detected")

        syntax_end = time.perf_counter()
        latencies.syntactic_complexity_ms = (syntax_end - syntax_start) * 1000

        # -------------------------------------------------------------
        # STAGE 4: Symbol & Structural Heuristics
        # -------------------------------------------------------------
        domain_start = time.perf_counter()
        math_or_code_tokens = [
            token.text for token in doc 
            if token.pos_ == "SYM" or token.text in {"=", "+", "-", "*", "/", ">", "<", "{", "}", "def", "lambda"}
        ]
        if math_or_code_tokens:
            matched_reasons.append(f"math_code_symbols({len(math_or_code_tokens)})")

        domain_end = time.perf_counter()
        latencies.domain_entity_eval_ms = (domain_end - domain_start) * 1000

        # -------------------------------------------------------------
        # FINAL ARBITRATION
        # -------------------------------------------------------------
        latencies.total_ms = (time.perf_counter() - total_start) * 1000

        # Decision Logic
        if len(lemma_matches) >= 2 or (lemma_matches and conditionals):
            return RoutingDecision(
                query=query,
                requires_thinking=True,
                effort_level=ReasoningEffort.HIGH,
                matched_reasons=matched_reasons,
                confidence=0.92,
                latencies=latencies,
            )

        if lemma_matches or conditionals or len(math_or_code_tokens) >= 2:
            return RoutingDecision(
                query=query,
                requires_thinking=True,
                effort_level=ReasoningEffort.LOW,
                matched_reasons=matched_reasons,
                confidence=0.82,
                latencies=latencies,
            )

        return RoutingDecision(
            query=query,
            requires_thinking=False,
            effort_level=ReasoningEffort.NONE,
            matched_reasons=matched_reasons or ["standard_direct_generation"],
            confidence=0.65,
            latencies=latencies,
        )

In [4]:
if __name__ == "__main__":
    router = SpacyComplexityRouter()

    test_queries = [
        "Hello!",
        "Translate this email to German: Meeting at 3pm.",
        "Calculate the eigenvalues if the matrix is upper triangular.",
        "Compare and contrast optimistic vs pessimistic locking under high contention.",
        "What is the capital of Japan?",
        "If x > 5 and y < 2, solve for the maximum value of 3x - 4y assuming integer constraints.",
    ]

    print(f"{'Query':<60} | {'Think?':<7} | {'Effort':<6} | {'spaCy (ms)':<10} | {'Total (ms)':<10}")
    print("-" * 102)

    for q in test_queries:
        res = router.analyze(q)
        display_q = (q[:57] + "...") if len(q) > 60 else q
        print(
            f"{display_q:<60} | "
            f"{str(res.requires_thinking):<7} | "
            f"{res.effort_level.value:<6} | "
            f"{res.latencies.spacy_doc_parse_ms:<10.3f} | "
            f"{res.latencies.total_ms:<10.3f}"
        )
        print(f"  └─ Reasons: {res.matched_reasons}\n")

Query                                                        | Think?  | Effort | spaCy (ms) | Total (ms)
------------------------------------------------------------------------------------------------------
Hello!                                                       | False   | none   | 1.148      | 1.154     
  └─ Reasons: ['salutation_token']

Translate this email to German: Meeting at 3pm.              | False   | none   | 1.644      | 1.655     
  └─ Reasons: ['direct_root_intent(translate)']

Calculate the eigenvalues if the matrix is upper triangular. | True    | high   | 1.664      | 1.694     
  └─ Reasons: ['reasoning_lemmas(calculate)', 'conditional_clause_detected']

Compare and contrast optimistic vs pessimistic locking un... | True    | high   | 1.535      | 1.557     
  └─ Reasons: ['reasoning_lemmas(compare,contrast)']

What is the capital of Japan?                                | False   | none   | 1.203      | 1.216     
  └─ Reasons: ['standard_direct_generation']